
# Task 3 – IB9AU – 2026

**Name**: qizhou lin  
**Task**: End-to-End Binary Classification - Loan Default Prediction using the German Credit Dataset.

---

### **Key Insights**
- Learned to **load and preprocess** real-world, unstructured data (space-separated, no headers).
- Applied **one-hot encoding** for categorical variables and **standardized** numerical features with `StandardScaler`.
- Built a custom **MLP (Multi-Layer Perceptron)** in PyTorch for binary classification.
- Used `BCELoss` and `Adam` optimizer in the **training loop**, and tracked loss across epochs.
- Evaluated the model and computed **test accuracy** to assess performance.



In [1]:
import pandas as pd
from google.colab import files
uploaded = files.upload()

Saving german.data.txt to german.data.txt


In [2]:
column_names = [
    "Status of existing checking account", "Duration in month", "Credit history", "Purpose",
    "Credit amount", "Savings account/bonds", "Present employment since",
    "Installment rate in percentage of disposable income", "Personal status and sex",
    "Other debtors / guarantors", "Present residence since", "Property", "Age in years",
    "Other installment plans", "Housing", "Number of existing credits at this bank", "Job",
    "Number of people being liable to provide maintenance for", "Telephone", "foreign worker",
    "Creditability"
]

df = pd.read_csv("german.data.txt", sep=' ', header=None, names=column_names)
df.head()

,Status of existing checking account,Duration in month,Credit history,Purpose,Credit amount,Savings account/bonds,Present employment since,Installment rate in percentage of disposable income,Personal status and sex,Other debtors / guarantors,...,Property,Age in years,Other installment plans,Housing,Number of existing credits at this bank,Job,Number of people being liable to provide maintenance for,Telephone,foreign worker,Creditability
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df['Creditability'] = df['Creditability'].replace({1: 1, 2: 0})
df_encoded = pd.get_dummies(df.drop(columns=['Creditability']))
X = df_encoded
y = df['Creditability']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df['Creditability'] = df['Creditability'].replace({1: 1, 2: 0})
df_encoded = pd.get_dummies(df.drop(columns=['Creditability']))
X = df_encoded
y = df['Creditability']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [5]:
import torch
from torch.utils.data import DataLoader, TensorDataset

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)


In [6]:
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, input_size):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

model = MLP(X_train.shape[1])


In [11]:
import torch.optim as optim

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 50
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")


Epoch 10/50, Loss: 10.0109
Epoch 20/50, Loss: 6.5975
Epoch 30/50, Loss: 3.1237
Epoch 40/50, Loss: 1.3754
Epoch 50/50, Loss: 0.5947


In [13]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        predicted = (outputs > 0.5).float()
        correct += (predicted == y_batch).sum().item()
        total += y_batch.size(0)

accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")


Test Accuracy: 0.7550
